# 第2日 ノートブック01：Python 入門

# Day 2 Notebook 01: Python for Absolute Beginners

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. `print()` で値を表示できる | 1. Display values with `print()` |
| 2. 変数に数値や文字列を代入できる | 2. Store numbers and text in variables |
| 3. 四則演算と比較演算を使える | 3. Use arithmetic and comparison operators |
| 4. True / False の意味を説明できる | 4. Explain what True and False mean |
| 5. エラーメッセージを読んで、自分で直せる | 5. Read an error message and fix it yourself |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 90 分 / 90 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第1日を終えていること。 / You have finished Day 1. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 演習 1〜3 のプログラム | Programs for exercises 1-3 |
| エラー記録 | Your error log |

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師の説明：Python の基本 | Instructor: Python basics |
| 09:00–10:30 | Part 1〜Part 3（表示・変数・計算） | Part 1-Part 3 (printing, variables, arithmetic) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4〜Part 6（比較・入力・エラー） | Part 4-Part 6 (comparison, input, errors) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：振り返りと午後の説明 | Instructor: review and briefing |
| 13:50–16:50 | 演習・提出物 | Exercises and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

---

## Part 1　表示する / Displaying values

#### 日本語：`print()` の基本
**日本語：** `print()` は、かっこの中の値を画面に表示します。

```python
print("こんにちは")
print(100.3)
print("幅 =", 100.3, "mm")
```

3 行目のように、`"..."` と数値をカンマで並べると、
**スペースで区切って** 続けて表示されます。

| 書き方 | 表示 |
|---|---|
| `print("abc")` | `abc` |
| `print(100.3)` | `100.3` |
| `print("abc", 100.3)` | `abc 100.3` |
| `print("abc" + "def")` | `abcdef`（つながる） |
| `print("abc" + 100.3)` | **エラー**（文字列と数値は足せない） |

**English:** `print()` shows the value inside the parentheses.

```python
print("hello")
print(100.3)
print("width =", 100.3, "mm")
```

When you list several values with commas, they are printed **separated by a space**.

| Code | Output |
|---|---|
| `print("abc")` | `abc` |
| `print(100.3)` | `100.3` |
| `print("abc", 100.3)` | `abc 100.3` |
| `print("abc" + "def")` | `abcdef` (joined) |
| `print("abc" + 100.3)` | **error** (text and number cannot be added) |
#### English: The basics of `print()`

In [ ]:
print("こんにちは / hello")
print(100.3)
print("幅 =", 100.3, "mm")
print("abc" + "def")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
こんにちは / hello
100.3
幅 = 100.3 mm
abcdef
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
hello
100.3
width = 100.3 mm
abcdef
```

---

## Part 2　変数 / Variables

#### 日本語：変数は「名前の付いた箱」
**日本語：** 変数は、値を入れておく **名前の付いた箱** です。

```python
sheet_width = 100.3
```

これは「`sheet_width` という箱に、`100.3` を入れる」という意味です。
数学の `=`（等しい）とは違います。

**変数名のルール**

- 英数字と `_`（アンダースコア）が使える
- 数字で始められない
- 大文字と小文字は区別される（`Width` と `width` は別物）
- 予約語（`if`, `for`, `class` など）は使えない

**良い名前の例：** `sheet_width`, `measured_thickness`, `defect_count`
**悪い名前の例：** `a`, `x1`, `data2`（後で読んだときに意味が分からない）

**English:** A variable is a **named box** holding a value.

```python
sheet_width = 100.3
```

means "put 100.3 into the box called `sheet_width`". It is not the mathematical "equals".

**Naming rules**

- letters, digits and `_` are allowed
- cannot start with a digit
- case sensitive (`Width` and `width` differ)
- reserved words (`if`, `for`, `class`, ...) are not allowed

**Good names:** `sheet_width`, `measured_thickness`, `defect_count`
**Bad names:** `a`, `x1`, `data2` — unreadable later.
#### English: A variable is a named box

In [ ]:
sheet_width_mm = 100.3
measured_thickness_mm = 2.04
defect_count = 3

print("シート幅 / sheet width      :", sheet_width_mm, "mm")
print("厚さ     / thickness        :", measured_thickness_mm, "mm")
print("欠陥数   / defect count     :", defect_count)

# 変数は後から書き換えられる / a variable can be reassigned
defect_count = defect_count + 1
print("1 枚追加した後 / after adding one:", defect_count)

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
シート幅 / sheet width      : 100.3 mm
厚さ     / thickness        : 2.04 mm
欠陥数   / defect count     : 3
1 枚追加した後 / after adding one: 4
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
sheet width      : 100.3 mm
thickness        : 2.04 mm
defect count     : 3
after adding one : 4
```

---

## Part 3　数値と計算 / Numbers and arithmetic

#### 日本語：数値の種類と四則演算
**日本語：** Python の数値には主に 2 種類あります。

| 種類 | 例 | 使う場面 |
|---|---|---|
| 整数 `int` | `3`, `-12`, `0` | 個数、画素数、番号 |
| 小数 `float` | `100.3`, `0.05`, `2.0` | 測定値、比率、時間 |

**演算子**

| 記号 | 意味 | 例 | 結果 |
|---|---|---|---|
| `+` | たす | `100.0 + 0.3` | `100.3` |
| `-` | ひく | `100.0 - 0.3` | `99.7` |
| `*` | かける | `1.2 * 100` | `120.0` |
| `/` | わる | `120 / 1.2` | `100.0` |
| `//` | わり算の整数部分 | `7 // 2` | `3` |
| `%` | わり算の余り | `7 % 2` | `1` |
| `**` | べき乗 | `2 ** 3` | `8` |

> **日本語：** `/` は必ず小数になります。`4 / 2` は `2.0` です（`2` ではありません）。
> 整数が欲しいときは `//` を使います。

**English:** Python has two main number types:

| Type | Examples | Used for |
|---|---|---|
| `int` | `3`, `-12`, `0` | counts, pixel indices, IDs |
| `float` | `100.3`, `0.05`, `2.0` | measurements, ratios, time |

| Operator | Meaning | Example | Result |
|---|---|---|---|
| `+` | add | `100.0 + 0.3` | `100.3` |
| `-` | subtract | `100.0 - 0.3` | `99.7` |
| `*` | multiply | `1.2 * 100` | `120.0` |
| `/` | divide | `120 / 1.2` | `100.0` |
| `//` | integer division | `7 // 2` | `3` |
| `%` | remainder | `7 % 2` | `1` |
| `**` | power | `2 ** 3` | `8` |

`/` always produces a float: `4 / 2` is `2.0`, not `2`. Use `//` when you need an integer.
#### English: Number types and the four operations

In [ ]:
nominal_mm = 100.0
measured_mm = 100.3

difference = measured_mm - nominal_mm
print("差 / difference      :", difference)
print("四捨五入 / rounded   :", round(difference, 2))

# 型を確認する / check the types
print("type(nominal_mm)     :", type(nominal_mm).__name__)
print("type(3)              :", type(3).__name__)
print("7 // 2               :", 7 // 2)
print("7 % 2                :", 7 % 2)
print("2 ** 3               :", 2 ** 3)

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
差 / difference      : 0.29999999999999716
四捨五入 / rounded   : 0.3
type(nominal_mm)     : float
type(3)              : int
7 // 2               : 3
7 % 2                : 1
2 ** 3               : 8
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
difference           : 0.29999999999999716
rounded              : 0.3
type(nominal_mm)     : float
type(3)              : int
7 // 2               : 3
7 % 2                : 1
2 ** 3               : 8
```

#### 日本語：小数の計算には誤差が出る
**日本語：** 上の結果で `0.30000000000001137` と出たことに気づきましたか？
`100.3 - 100.0` が正確に `0.3` にならないのです。

これは Python の不具合ではなく、**コンピュータが小数を 2 進数で表す** ために起きる、
避けられない性質です。0.1 も 0.3 も、2 進数では正確に表せません。

**製造検査での意味：** しきい値の比較では、この誤差が問題になることがあります。
`round()` で丸める、または「差が 0.5 以下か」のように **不等号** で比べるのが安全です。

**English:** Notice `0.30000000000001137` above. `100.3 - 100.0` is not exactly `0.3`.

This is not a Python bug: computers store decimals in binary, and 0.1 / 0.3 cannot be represented
exactly. It is unavoidable.

**Why it matters for inspection:** threshold comparisons can be affected. Round with `round()`,
or compare with inequalities ("is the difference <= 0.5?") instead of equality.
#### English: Floating point has rounding error

In [ ]:
print("0.1 + 0.2 =", 0.1 + 0.2)
print("等しいか / is it 0.3? :", 0.1 + 0.2 == 0.3)
print("round(0.1 + 0.2, 10) == 0.3 :", round(0.1 + 0.2, 10) == 0.3)
print()
print("安全な比較 / safe comparison:", abs((0.1 + 0.2) - 0.3) < 1e-9)

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
0.1 + 0.2 = 0.30000000000000004
等しいか / is it 0.3? : False
round(0.1 + 0.2, 10) == 0.3 : True

安全な比較 / safe comparison: True
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
0.1 + 0.2 = 0.30000000000000004
is it 0.3? : False
round(0.1 + 0.2, 10) == 0.3 : True

safe comparison: True
```

#### 日本語：文字列
**日本語：** 文字列は `"..."` で囲んだ文字の並びです。`'...'` でも同じです。

```python
product_name = "PVC sheet"
grade = 'A'
```

**よく使う操作**

| 操作 | 例 | 結果 |
|---|---|---|
| 連結 | `"PVC" + " sheet"` | `PVC sheet` |
| 繰り返し | `"ab" * 3` | `ababab` |
| 長さ | `len("PVC")` | `3` |
| 一部を取り出す | `"PVC sheet"[0:3]` | `PVC` |
| 大文字に | `"pvc".upper()` | `PVC` |

> **日本語：** 番号は **0 から** 始まります。`[0:3]` は「0 番目から 3 番目の手前まで」です。

**English:** A string is a sequence of characters in quotes: `"..."` or `'...'`.

| Operation | Example | Result |
|---|---|---|
| concatenate | `"PVC" + " sheet"` | `PVC sheet` |
| repeat | `"ab" * 3` | `ababab` |
| length | `len("PVC")` | `3` |
| slice | `"PVC sheet"[0:3]` | `PVC` |
| upper case | `"pvc".upper()` | `PVC` |

Indexing starts at **0**, and `[0:3]` means "from index 0 up to but not including 3".
#### English: Strings

In [ ]:
product_name = "PVC sheet"
grade = "A"

print(product_name, "grade", grade)
print("length of product_name:", len(product_name))
print("first three characters:", product_name[0:3])
print("repeated:", "ab" * 3)
print("upper:", product_name.upper())

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
PVC sheet grade A
length of product_name: 9
first three characters: PVC
repeated: ababab
upper: PVC SHEET
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
PVC sheet grade A
length of product_name: 9
first three characters: PVC
repeated: ababab
upper: PVC SHEET
```

---

## Part 4　比較と真偽値 / Comparison and booleans

#### 日本語：比較演算子
**日本語：** 比較の結果は `True`（正しい）か `False`（正しくない）になります。
この 2 つを **真偽値（ブール値）** と呼びます。

| 記号 | 意味 | 例 | 結果 |
|---|---|---|---|
| `==` | 等しい | `100.0 == 100.0` | `True` |
| `!=` | 等しくない | `1 != 2` | `True` |
| `<` | より小さい | `1 < 2` | `True` |
| `<=` | 以下 | `2 <= 2` | `True` |
| `>` | より大きい | `1 > 2` | `False` |
| `>=` | 以上 | `2 >= 3` | `False` |

> **日本語：** **`=` と `==` は別物** です。`=` は代入、`==` は比較。
> ここは初心者が最もよく間違えるところです。

**English:** A comparison produces `True` (correct) or `False` (incorrect). These are **booleans**.

| Operator | Meaning | Example | Result |
|---|---|---|---|
| `==` | equal | `100.0 == 100.0` | `True` |
| `!=` | not equal | `1 != 2` | `True` |
| `<` | less than | `1 < 2` | `True` |
| `<=` | less or equal | `2 <= 2` | `True` |
| `>` | greater than | `1 > 2` | `False` |
| `>=` | greater or equal | `2 >= 3` | `False` |

**`=` and `==` are different**: `=` assigns, `==` compares. This is the single most common
beginner mistake.
#### English: Comparison operators

In [ ]:
measured_mm = 100.3
nominal_mm = 100.0
tolerance_mm = 0.5

difference = measured_mm - nominal_mm
within_tolerance = abs(difference) <= tolerance_mm

print("差 / difference      :", round(difference, 3))
print("規格内か / in spec?  :", within_tolerance)
print("型 / type            :", type(within_tolerance).__name__)
print()
print("100.0 == 100.0 :", 100.0 == 100.0)
print("1 != 2         :", 1 != 2)
print("2 >= 3         :", 2 >= 3)

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
差 / difference      : 0.3
規格内か / in spec?  : True
型 / type            : bool

100.0 == 100.0 : True
1 != 2         : True
2 >= 3         : False
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
difference  : 0.3
in spec?    : True
type        : bool

100.0 == 100.0 : True
1 != 2         : True
2 >= 3         : False
```

---

## Part 5　キーボード入力 / Reading input from the keyboard

#### 日本語：`input()` の使い方と注意点
**日本語：** `input()` は、キーボードから 1 行読み取ります。**必ず文字列** として返します。

```python
name = input("名前を入力してください: ")
print("こんにちは", name)
```

数値として使いたいときは、**変換が必要** です。

```python
text = input("幅を入力してください: ")
width = float(text)          # 文字列 -> 小数
```

**この研修で `input()` を使わない理由**

| 理由 | 説明 |
|---|---|
| 自動実行できない | ノートブックを一括実行すると、入力待ちで止まる |
| 記録が残らない | 何を入力したか分からなくなる |
| 誤入力に弱い | 数字以外を入れるとエラーになる |

**日本語：** 代わりに、**セルの中に値を書いて** 実行します。値を変えて何度でも試せます。

**English:** `input()` reads one line from the keyboard and always returns **text**.

```python
name = input("your name: ")
print("hello", name)
```

To use a number you must convert it:

```python
text = input("width: ")
width = float(text)          # text -> float
```

**Why this course avoids `input()`**

| Reason | Explanation |
|---|---|
| cannot run unattended | a batch run stops and waits |
| no record | you cannot see later what was typed |
| fragile | typing letters instead of digits raises an error |

Instead, **write the value in the cell** and run it. Change it and run again as often as you like.
#### English: How `input()` works, and its warnings

In [ ]:
# 見本：実行しないでください（入力待ちになります）
# Example only: do not run (it waits for the keyboard)
#
# text = input("幅を入力してください / width: ")
# width = float(text)
# print("入力された幅 / you typed:", width, "mm")
#
# 代わりに、値をセルに書いて実行します / instead, put the value in the cell:
measured_width_text = "100.3"
measured_width = float(measured_width_text)
print("measured_width =", measured_width, type(measured_width).__name__)

---

## Part 6　エラーとデバッグ / Errors and debugging

#### 日本語：よく出るエラー 4 種類
**日本語：** エラーは「失敗」ではなく「情報」です。名前で原因が分かります。

| エラー | 意味 | よくある原因 |
|---|---|---|
| `SyntaxError` | 文法がおかしい | `if x > 1` の `:` 忘れ、括弧の閉じ忘れ |
| `NameError` | その名前は未定義 | 変数名の打ち間違い、前のセルを実行していない |
| `TypeError` | 型が合わない | `"abc" + 100`（文字列と数値） |
| `ValueError` | 値がおかしい | `float("abc")`（数値でない文字列） |
| `IndexError` | 範囲外 | 8 個の配列に `[10]` でアクセス |

**エラーメッセージの読み方（3 ステップ）**

1. **最後の行**：エラーの種類（`NameError` など）
2. **その上の行**：何をしようとして失敗したか
3. **`---->` の行**：プログラムの何行目か（Jupyter では `line 3` など）

**English:** An error message is information. The name tells you the cause.

| Error | Meaning | Typical cause |
|---|---|---|
| `SyntaxError` | invalid syntax | missing `:` after `if`, unbalanced brackets |
| `NameError` | name is not defined | typo in a variable name, earlier cell not run |
| `TypeError` | wrong type | `"abc" + 100` |
| `ValueError` | wrong value | `float("abc")` |
| `IndexError` | out of range | index `10` on an 8-element array |

**Reading an error in three steps**

1. the **last line**: the error type
2. the line above: what failed
3. the `---->` line: which line of your program
#### English: The four errors you will meet most

In [ ]:
# わざとエラーを起こす / create errors on purpose
examples = {
    "NameError": "print(not_defined_variable)",
    "TypeError": "print('abc' + 100)",
    "ValueError": "print(float('abc'))",
    "IndexError": "print([1, 2, 3][10])",
}

for label, statement in examples.items():
    print("=" * 40)
    print("例 / example:", label)
    try:
        exec(statement)
    except Exception as error:
        print("  ", type(error).__name__, ":", error)
        print("   -> これは想定どおりのエラーです / this error is intentional")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
例 / example: NameError
   NameError : name 'not_defined_variable' is not defined
...
例 / example: TypeError
   TypeError : can only concatenate str (not "int") to str
...
例 / example: ValueError
   ValueError : could not convert string to float: 'abc'
...
例 / example: IndexError
   IndexError : list index out of range
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
example: NameError
   NameError : name 'not_defined_variable' is not defined
...
example: TypeError
   TypeError : can only concatenate str (not "int") to str
...
example: ValueError
   ValueError : could not convert string to float: 'abc'
...
example: IndexError
   IndexError : list index out of range
```

#### 日本語：デバッグの手順
**日本語：** エラーが出たら、次の順番で確認してください。

1. **どのセルか** を確認する（`In [ ]` の番号）
2. **メッセージの最後の行** を読む
3. **直前のセルまで実行したか** を確認する
4. **疑わしい値を `print()` する** — これが最も確実な方法です
5. 直したら、**そのセルだけ** 実行し直す

**English:** When an error appears:

1. note **which cell** it happened in
2. read the **last line** of the message
3. check whether you ran **all preceding cells**
4. **`print()` the suspicious value** — the most reliable technique
5. after fixing, re-run **only that cell**
#### English: A debugging procedure

In [ ]:
# デバッグ実演：あやしい値を表示する / debugging demo: print suspicious values
raw_text = "100.3 "
cleaned = raw_text.strip()

print("raw    :", repr(raw_text))
print("cleaned:", repr(cleaned))

try:
    print("value  :", float(cleaned))
except ValueError as error:
    print("変換に失敗 / conversion failed:", error)

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：自己紹介を表示する
**日本語：** 変数に「名前」「所属」「今日の学習目標」を入れて、`print()` で 3 行表示してください。
日本語 1 行、英語 2 行の構成にしてください。

**English:** Store your name, your department and today's learning goal in variables, then print
three lines: one in Japanese, two in English.
#### English: Exercise 1 (basic): print an introduction
Done when the output shows your real information, not the placeholder text.

In [ ]:
# 演習 1 / Exercise 1
name = ""
department = ""
goal = ""

print("")
print("")
print("")

#### 日本語：演習 2（中級）：幅・長さ・面積を計算する
**日本語：** 幅 1,300 mm、長さ 2,500 mm のシートがあります。

1. 面積を mm² で表示してください
2. 面積を m² に変換して表示してください（1 m² = 1,000,000 mm²）
3. 1 枚あたり 0.35 mm 厚、密度 1.4 g/cm³ として、重さを kg で概算してください

**English:** A sheet is 1,300 mm wide and 2,500 mm long.

1. print the area in mm²
2. convert it to m² (1 m² = 1,000,000 mm²)
3. estimate the mass in kg for 0.35 mm thickness and 1.4 g/cm³ density

Hint: mass = volume x density; convert mm to cm first.
#### English: Exercise 2 (intermediate): compute width, length and area
Done when the three values are printed with units.

In [ ]:
# 演習 2 / Exercise 2
width_mm = 1300.0
length_mm = 2500.0
thickness_mm = 0.35
density_g_per_cm3 = 1.4

area_mm2 = 0.0        # <- 計算してください / compute
area_m2 = 0.0         # <- 計算してください / compute
mass_kg = 0.0         # <- 計算してください / compute

print("面積 / area      :", area_mm2, "mm2")
print("面積 / area      :", area_m2, "m2")
print("重さ / mass      :", mass_kg, "kg")

#### 日本語：演習 3（発展）：エラーを直す
**日本語：** 下のセルには **わざと 3 つの誤り** が入ったプログラムが入っています。
まずそのまま実行して、**3 つのエラーメッセージ** を読んでください。

そのあと、次の「解答用」セルに **正しく直したプログラム** を書いてください。
直したら、何が原因だったかをコメントに書いてください。

**English:** The next cell holds a program with **three deliberate mistakes**. Run it and read the
**three error messages**.

Then write the **corrected** program in the "answer" cell below, and record the cause of each
mistake as a comment.
#### English: Exercise 3 (challenge): fix the errors
Done when the answer cell prints the OK/NG result and three cause comments are written.

In [ ]:
buggy_program = """
measured = "100.3"
nominal = 100.0
tolerance = 0.5

difference = measured - nominal
if abs(difference) <= tolerance
    print("OK")
else:
    print("NG")
"""

print("--- わざとの誤りを含むプログラム / program with deliberate mistakes ---")
try:
    exec(compile(buggy_program, "<exercise>", "exec"))
except (SyntaxError, TypeError, NameError) as error:
    print(type(error).__name__, ":", error)
    print()
    print("エラーは 1 つずつ直します。まずは 7 行目（if 文）から。")
    print("Fix one error at a time, starting with line 7 (the if statement).")

In [ ]:
# 演習 3 解答用 / Exercise 3 answer cell
# 上と同じ内容を、正しく直して書いてください / rewrite the program correctly

measured = 0.0        # <- 数値にしてください / use a number
nominal = 100.0
tolerance = 0.5

difference = measured - nominal
# ここに正しい if 文を書いてください / write the correct if statement

# 原因 1 / cause 1:
# 原因 2 / cause 2:
# 原因 3 / cause 3:

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. `=` と `==` の違いを説明してください。
2. `100.3 - 100.0` が `0.3` にならないのはなぜですか。
3. `print(a, b)` と `print(a + b)` の違いは何ですか（`a` が文字列、`b` が数値のとき）。
4. `NameError` が出たとき、最初に確認することは何ですか。
5. 変数名を `x1` ではなく `measured_width_mm` にする利点は何ですか。

**English:**

1. Explain the difference between `=` and `==`.
2. Why is `100.3 - 100.0` not exactly `0.3`?
3. What is the difference between `print(a, b)` and `print(a + b)` when `a` is text and `b` a number?
4. When you see `NameError`, what do you check first?
5. Why name a variable `measured_width_mm` instead of `x1`?
#### English: Understanding questions
Model answers: `docs/instructor/day02_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `SyntaxError: invalid syntax` | `SyntaxError: invalid syntax` | `:` の忘れ、括弧の閉じ忘れ、全角スペースが最も多い原因です。行番号を確認してください。 | Most often a missing `:`, an unclosed bracket, or a full-width Japanese space. |
| `NameError: name 'x' is not defined` | `NameError: name 'x' is not defined` | 変数名の綴り、または前のセルを実行していないことが原因です。 | A typo in the name, or an earlier cell was not run. |
| `IndentationError` | `IndentationError` | 行頭のスペースが揃っていません。`if` の中は 4 文字下げます。タブとスペースを混ぜないでください。 | Indentation is inconsistent. Use 4 spaces inside `if`; never mix tabs and spaces. |
| `TypeError: can only concatenate str (not "int") to str` | `TypeError: can only concatenate str (not "int") to str` | 文字列と数値を `+` で足そうとしています。`print("a", 1)` か `f"a{1}"` を使ってください。 | You are adding text and a number with `+`. Use `print("a", 1)` or an f-string. |
| 計算結果が `0.30000000000001137` になる | The result is `0.30000000000001137` | 小数の誤差です。`round()` で丸めるか、不等号で比較してください。 | Floating point error. Use `round()` or compare with inequalities. |
| 日本語が `??` や `□□` になる | Japanese shows as `??` or `□□` | コンソールの文字コードの可能性があります。ノートブック上では通常起きません。 | Usually a console codepage issue; it does not normally happen in the notebook. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 6 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-6 and ran every code cell
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 演習 3 の 3 つのエラーを自分で直せた
      - [ ] Fixed all three errors in exercise 3
- [ ] `=` と `==` の違いを説明できる
      - [ ] Can explain `=` versus `==`
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] エラー記録を残した
      - [ ] Wrote an error log
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Python 公式チュートリアル（日本語）](https://docs.python.org/ja/3/tutorial/)
- [Python 組み込み型](https://docs.python.org/ja/3/library/stdtypes.html)
- [浮動小数点演算の性質](https://docs.python.org/ja/3/tutorial/floatingpoint.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** `print(f"{0.1:.20f}")` を実行して、0.1 が内部でどう表されているか見てください。

**English:** Run `print(f"{0.1:.20f}")` and see how 0.1 is actually stored.